# From Function Calling to the Agent Loop

The full code of [From Function Calling to the Agent Loop](https://residual-stream.lvtan71.workers.dev/posts/function-calling-to-agent-loop/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/function-calling-to-agent-loop) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/function-calling-to-agent-loop.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 19 min end to end.
Downloads Qwen2.5-1.5B-Instruct and Qwen2.5-3B-Instruct (about 9 GB together) and MiniLM.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q numpy "torch==2.14.*" "transformers==5.18.*" sentence-transformers huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "function-calling-to-agent-loop"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

In [ ]:
!tar xzf blog-a7da2a2.tar.gz

## `chat_template.py`

One tool-calling exchange rendered through Qwen2.5's chat template: the tool schema in the system prompt, the call between `<tool_call>` tags, the result in `<tool_response>` tags.

In [ ]:
!{sys.executable} chat_template.py

The post printed:

```text
<|im_start|>system
You answer questions about a technical blog, using its tools.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_posts", "description": "Search the blog.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
How can a maximum and a running sum be updated incrementally?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "search_posts", "arguments": {"query": "online softmax running maximum"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
[{"post": "flashattention-explained", "heading": "The online softmax", "snippet": "Keep a running maximum m..."}]
</tool_response><|im_end|>
<|im_start|>assistant
```

## `transcript_cost.py`

Tokens an agent's transcript reads over 5 to 50 turns, and the input cost with and without prompt caching.

In [ ]:
!{sys.executable} transcript_cost.py

The post printed:

```text
prefix 5,000 tokens, 1,500 new per turn, $3 per million input tokens
 turns  final context  tokens read in all  uncached   cached
     5       11,000              40,000   $   0.12  $  0.05
    10       18,500             117,500   $   0.35  $  0.10
    20       33,500             385,000   $   1.16  $  0.23
    50       78,500           2,087,500   $   6.26  $  0.90
```

## `both_models.py`

Runs `agent.py 1.5B`, then `agent.py 3B`.

In [ ]:
!{sys.executable} both_models.py

The post printed:

```text
446 sections; 30 questions; Qwen2.5-1.5B-Instruct, greedy

                                  found  cites any  cites right  model calls  tool calls  last context  tokens read  no tool  malformed
one retrieval, one answer            23         28           17          1.0         1.0           427          427        0          0
agent loop                            0         26            0          1.0         0.0           286          286       30          0
agent, 'always search first'          0         25            0          1.0         0.0           309          309       30          0
agent, first call forced             18          1            2          2.0         1.0           710        1,021        0          0
446 sections; 30 questions; Qwen2.5-3B-Instruct, greedy

                                  found  cites any  cites right  model calls  tool calls  last context  tokens read  no tool  malformed
one retrieval, one answer            23         30           23          1.0         1.0           427          427        0          0
agent loop                            4         29            3          1.2         0.2           371          428       24          0
agent, 'always search first'          9         28            3          1.4         0.4           491          624       17          0
agent, first call forced             21         24           10          2.0         1.0           720        1,030        0          0
```

## `rescore.py`

Rescores the logged answers: a citation counts only when the source line names exactly one of the blog's posts, the right one. Reads `agent_log_1.5B.jsonl` and `agent_log_3B.jsonl`.

In [ ]:
!{sys.executable} rescore.py

The post printed:

```text
Qwen2.5-1.5B-Instruct
                                    found  cites right   source line: none / one real post / several / placeholder or invented
  one retrieval, one answer            23           17      2 / 26 /  0 /  2
  agent loop                            0            0      4 /  0 /  0 / 26
  agent, 'always search first'          0            0      5 /  0 /  0 / 25
  agent, first call forced             18            0     29 /  0 /  1 /  0
Qwen2.5-3B-Instruct
                                    found  cites right   source line: none / one real post / several / placeholder or invented
  one retrieval, one answer            23           23      0 / 30 /  0 /  0
  agent loop                            4            3      1 /  4 /  0 / 25
  agent, 'always search first'          9            3      2 /  3 /  1 / 24
  agent, first call forced             21           10      6 / 15 /  1 /  8
```